<a href="https://colab.research.google.com/github/muntakson/omics-atlas-workshop/blob/main/notebooks/05_single_cell.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧬 Workshop 5 — Single-cell RNA-seq with Scanpy

**Companion to [Chapter 5: Current Best Practices in Single-Cell RNA-seq Analysis](https://omics.iotok.org/chapters/scrnaseq-best-practices)**.

You will run the full single-cell workflow from Chapter 5 — QC → normalization → feature selection → PCA → UMAP → clustering → marker genes — on a **real** dataset with **Scanpy**.

### The dataset — 10x Genomics PBMC 3k
~2,700 peripheral-blood mononuclear cells (PBMCs) from a healthy donor, sequenced with 10x Chromium. It is the canonical scRNA-seq teaching dataset and ships with Scanpy.

> Run each cell with **Shift+Enter**. Look for the ✏️ **Your turn** exercises.

## 1. Install & load
Scanpy (and `leidenalg` for clustering) aren't in the Colab base image — install them (≈40 s).

In [ ]:
!pip install -q scanpy leidenalg igraph

In [ ]:
import scanpy as sc
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=70, facecolor='white')

adata = sc.datasets.pbmc3k()   # real 10x PBMC data, downloaded once
adata.var_names_make_unique()
print(adata)   # AnnData: cells (obs) x genes (var)

An **AnnData** object holds the count matrix plus per-cell (`obs`) and per-gene (`var`) annotation. Right now it's raw UMI counts for ~2,700 cells × ~32,000 genes.

## 2. Quality control
Chapter 5 → *Quality Control*. We flag mitochondrial genes (high % = stressed/dying cells) and compute per-cell metrics.

In [ ]:
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
sc.pl.violin(adata, ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
             jitter=0.4, multi_panel=True)

Filter out low-quality cells (too few genes, or high mitochondrial fraction) and genes seen in almost no cells.

In [ ]:
sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)
adata = adata[adata.obs.n_genes_by_counts < 2500, :]
adata = adata[adata.obs.pct_counts_mt < 5, :].copy()
print(adata.n_obs, 'cells remain')

## 3. Normalize
Chapter 5 → *Normalization*. Scale each cell to the same total count, then `log1p` transform.

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

## 4. Feature selection — highly variable genes
Chapter 5 → *Feature Selection*. We keep only the most variable genes for downstream steps (they carry the biological signal).

In [ ]:
sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
sc.pl.highly_variable_genes(adata)
adata.raw = adata                      # keep full log-normalized data
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)       # z-score genes

## 5. Dimensionality reduction — PCA & UMAP
Chapter 5 → *Dimensionality Reduction and Visualization*. PCA first, then a neighbor graph, then UMAP for 2-D visualization.

In [ ]:
sc.tl.pca(adata, svd_solver='arpack')
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
sc.tl.umap(adata)

## 6. Clustering
Chapter 5 → *Clustering*. The **Leiden** algorithm partitions the neighbor graph into cell communities.

In [ ]:
sc.tl.leiden(adata, resolution=1.0, flavor='igraph', n_iterations=2, directed=False)
print(adata.obs['leiden'].value_counts())
sc.pl.umap(adata, color='leiden', legend_loc='on data', title='Leiden clusters')

Overlay a few canonical marker genes to see which clusters are which cell types (CST3 = monocytes/DCs, NKG7 = NK/cytotoxic, MS4A1 = B cells).

In [ ]:
sc.pl.umap(adata, color=['CST3', 'NKG7', 'MS4A1'], ncols=3)

## 7. Marker genes per cluster
Chapter 5 → *Cluster Annotation and Marker Genes*. Rank the genes that define each cluster with a Wilcoxon test.

In [ ]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=15, sharey=False)

In [ ]:
import pandas as pd
# top 5 marker genes per cluster as a table
pd.DataFrame(adata.uns['rank_genes_groups']['names']).head(5)

### ✏️ Your turn
**MS4A1** marks B cells. Colour the UMAP by another marker — try **PPBP** (platelets) or **LYZ** (monocytes) — and see which cluster lights up.
*Hint: `sc.pl.umap(adata, color='LYZ')`.*

In [ ]:
# your code here


<details><summary>▶ Solution</summary>

In [ ]:
sc.pl.umap(adata, color=['LYZ', 'PPBP'], ncols=2)

</details>

## 🎓 You finished the single-cell workshop!
From raw 10x counts you did QC, normalization, feature selection, PCA/UMAP, Leiden clustering, and marker-gene identification — the complete Chapter 5 workflow.

Back to **[Omics Atlas](https://omics.iotok.org)** · **[Chapter 5](https://omics.iotok.org/chapters/scrnaseq-best-practices)**.